
# MÉTODOS NUMÉRICOS  
## Erros, Ponto Flutuante, Subtraç\tilde{a}o Catastrófica e Estabilidade — Notebook modular

Este notebook foi desenhado como **material permanente** e **modular**.

### Como usar (sugest\tilde{a}o)
- **Trilha A (aula de 2h, essencial)**: Módulos 0 → 6 + Exercícios 1–4.
- **Trilha B (com aprofundamento)**: incluir Módulos 7–9 (condicionamento simbólico, backward error, “erro do erro”).
- **Trilha C (opcional, acumulaç\tilde{a}o de erros)**: Módulo 10 (somas: ingênua vs Kahan vs pareada).

> Convenç\tilde{a}o: quando possível, use alta precis\tilde{a}o (`decimal`/`mpmath`) como “referência” — n\tilde{a}o como “verdade absoluta”.


### Submiss\tilde{a}o

Os estudantes dever\tilde{a}o **submeter um caderno Colab com as respostas**.

Ao longo do material, há células de código e células Markdown indicadas como espaço de resposta.  
As respostas devem ser escritas diretamente nessas células.



## Sumário de módulos

- **Módulo 0** — Setup e utilitários  
- **Módulo 1** — Ponto flutuante: primeira fricç\tilde{a}o com a realidade  
- **Módulo 2** — Medidas de erro: absoluto, relativo e “dígitos corretos”  
- **Módulo 3** — Investigaç\tilde{a}o: subtraç\tilde{a}o de números próximos  
- **Módulo 4** — Epsilon da máquina e arredondamento  
- **Módulo 5** — Alta precis\tilde{a}o como referência (Decimal/mpmath)  
- **Módulo 6** — Exemplo clássico: fórmula quadrática e reformulaç\tilde{a}o estável  
- **Módulo 7** — Condicionamento (inclui análise simbólica)  
- **Módulo 8** — Estabilidade algorítmica: forward/backward error (ideia)  
- **Módulo 9** — O “erro do erro”: por que nem o erro calculado é exato  
- **Módulo 10 (Opcional)** — Acumulaç\tilde{a}o de erros: soma ingênua vs Kahan vs pareada  
- **Exercícios finais** — com espaço para resposta



---
# Módulo 0 — Setup e utilitários


In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from decimal import Decimal, getcontext
getcontext().prec = 80  # ajuste conforme necessário

def erro_abs(x, x_aprox):
    return abs(x - x_aprox)

def erro_rel(x, x_aprox):
    return abs(x - x_aprox) / abs(x) if x != 0 else float('inf')

def digitos_corretos(x, x_aprox):
    # dígitos corretos ~ -log10(erro relativo), quando faz sentido
    er = erro_rel(x, x_aprox)
    return float('inf') if er == 0 else -math.log10(er)



---
# Módulo 1 — Ponto flutuante: primeira fricç\tilde{a}o com a realidade

**Pergunta-guia:** o computador “faz conta” com números reais?


In [ ]:
0.1 + 0.2

In [ ]:
format(0.1, ".25f")


**Investigue:**
1. Por que n\tilde{a}o aparece exatamente 0.3?
2. O erro é grande ou pequeno em termos absolutos? E em termos relativos?
3. Qual hipótese você faria sobre a representaç\tilde{a}o interna?



---
# Módulo 2 — Medidas de erro (e limitações)

Definições:
- Erro absoluto: \(E_a = |x - \tilde{x}|\)
- Erro relativo: \(E_r = |x - \tilde{x}| / |x|\) (cuidado quando \(x\approx 0\))

**Pergunta-guia:** qual medida faz mais sentido em cada escala?


In [ ]:

x = math.pi
x_aprox = 3.14

Ea = erro_abs(x, x_aprox)
Er = erro_rel(x, x_aprox)
Ea, Er



**Discuss\tilde{a}o curta (para a turma):**
- Erro absoluto depende de escala.
- Erro relativo “normaliza” pela grandeza do valor verdadeiro.
- Se \(x=0\), erro relativo n\tilde{a}o é uma boa medida (divis\tilde{a}o por zero / “explode”).



---
# Módulo 3 — Investigaç\tilde{a}o: subtraç\tilde{a}o de números próximos

**Pergunta-guia:** pequenos erros nos dados podem virar um grande erro no resultado?


In [ ]:

a = 1.23456789012345
b = 1.23456788912345

d_real = a - b

a_til = a + 1e-15
b_til = b - 1e-15
d_aprox = a_til - b_til

Ea = erro_abs(d_real, d_aprox)
Er = erro_rel(d_real, d_aprox)

d_real, d_aprox, Ea, Er



**Investigue:**
1. O erro absoluto parece grande?
2. E o erro relativo?
3. O que há de especial em subtrair valores muito próximos?


In [ ]:

ks = range(1, 17)
erros_rel = []

for k in ks:
    a = 1.0
    b = 1.0 - 10**(-k)

    # perturbação (simulando “erro nos dados”)
    a_til = a + 1e-16
    b_til = b - 1e-16

    d_real = a - b
    d_aprox = a_til - b_til

    erros_rel.append(erro_rel(d_real, d_aprox))

plt.semilogy(list(ks), erros_rel, marker='o')
plt.xlabel("k (em b = 1 - 10^{-k})")
plt.ylabel("Erro relativo em d = a - b")
plt.title("Amplificação de erro na subtração de números próximos")
plt.grid(True)
plt.show()


In [ ]:

for k, er in zip(ks, erros_rel):
    a = 1.0
    b = 1.0 - 10**(-k)
    d_real = a - b
    # estimativa de “dígitos corretos” (quando faz sentido)
    print(f"k={k:2d}  d={d_real:.1e}  dígitos corretos ≈ {-math.log10(er):.2f}")



**Nome do fenômeno (agora sim): Subtraç\tilde{a}o catastrófica**  
Cancelamento de dígitos significativos → erro relativo pode explodir.



---
# Módulo 4 — Epsilon da máquina e arredondamento

**Pergunta-guia:** qual é a “granularidade” do float perto de 1?


In [ ]:

np.finfo(float)


In [ ]:

eps = np.finfo(float).eps
eps, (1.0 + eps != 1.0), (1.0 + eps/2 == 1.0)



Modelo clássico de ponto flutuante (idealizado):

\\[ fl(x \circ y) = (x \circ y)(1 + \delta), \quad |\delta| \le \varepsilon_{mach} \\]

Isso n\tilde{a}o “remove” o erro; fornece um modo de **raciocinar e limitar** o erro.



---
# Módulo 5 — Alta precis\tilde{a}o como referência (Decimal / mpmath)

**Pergunta-guia:** o float está errado ou apenas limitado?


In [ ]:

# Decimal (alta precisão)
getcontext().prec = 80

aD = Decimal("1.23456789012345")
bD = Decimal("1.23456788912345")

dD = aD - bD
dF = float(aD) - float(bD)

dD, dF


In [ ]:

# mpmath (opcional, geralmente disponível no Colab)
import mpmath as mp
mp.mp.dps = 80

aM = mp.mpf("1.23456789012345")
bM = mp.mpf("1.23456788912345")

float(aM - bM), (aM - bM)



---
# Módulo 6 — Exemplo clássico: fórmula quadrática

**Pergunta-guia:** a fórmula “correta” pode ser numericamente ruim?


In [ ]:

a = 1
b = 10**8
c = 1

x1_dir = (-b + math.sqrt(b*b - 4*a*c)) / (2*a)
x2_dir = (-b - math.sqrt(b*b - 4*a*c)) / (2*a)

x1_dir, x2_dir



Uma reformulaç\tilde{a}o estável usa:
- calcular uma raiz sem cancelamento,
- obter a outra por \(x_2 = c/(a x_1)\).


In [ ]:

x1_est = (-b - math.sqrt(b*b - 4*a*c)) / (2*a)
x2_est = c / (a * x1_est)
x1_est, x2_est


In [ ]:

def residuo(a,b,c,x):
    return a*x*x + b*x + c

res = {
    "direta x1": residuo(a,b,c,x1_dir),
    "direta x2": residuo(a,b,c,x2_dir),
    "estável x1": residuo(a,b,c,x1_est),
    "estável x2": residuo(a,b,c,x2_est),
}
res



---
# Módulo 7 — Condicionamento (com análise simbólica)

Para \\(f(a,b) = a-b\\), uma heurística de condicionamento relativo é:

\\[ \kappa(a,b) \approx \frac{|a| + |b|}{|a-b|}. \\]

**Leitura:** se \\(a \to b\\), ent\tilde{a}o \\(|a-b| \to 0\\) e \\(\kappa\\) cresce muito → o problema fica *mal condicionado*.


In [ ]:

import sympy as sp

a, b, da, db = sp.symbols('a b da db', positive=True, real=True)
f = a - b

# erro absoluto (primeira ordem): |da - db|, omitindo valor absoluto para álgebra simbólica
df = sp.diff(f, a)*da + sp.diff(f, b)*db  # = 1*da + (-1)*db
df_simpl = sp.simplify(df)

df_simpl



Aproximaç\tilde{a}o de primeira ordem (diferencial total):

\\[ \Delta f \approx \frac{\partial f}{\partial a}\Delta a + \frac{\partial f}{\partial b}\Delta b = \Delta a - \Delta b. \\]

Se \\(|\Delta a| \le \eta |a|\\) e \\(|\Delta b| \le \eta |b|\\), ent\tilde{a}o

\\[ |\Delta f| \le \eta(|a| + |b|). \\]

Dividindo por \\(|f| = |a-b|\\):

\\[ \frac{|\Delta f|}{|f|} \le \eta \cdot \frac{|a| + |b|}{|a-b|}. \\]

Isso é uma cota (limitante) para o erro relativo: mesmo sem saber o erro exato, conseguimos **garantir um limite superior**.



---
# Módulo 8 — Estabilidade algorítmica (ideia)

- **Condicionamento** é propriedade do problema.
- **Estabilidade** é propriedade do algoritmo.

**Ideia (informal):**
Um algoritmo é *estável* se o erro que ele produz é comparável ao erro inevitável de arredondamento (e ao condicionamento do problema),
em vez de amplificar erros desnecessariamente.

Dois algoritmos podem resolver o mesmo problema:
- um pode ser estável (erro “controlado”),
- outro instável (erro “explodindo”).



Noções úteis (sem formalismo pesado nesta aula):

- **Forward error:** \\(|\\tilde{x} - x|\\)
- **Backward error:** o resultado computado \\(\\tilde{x}\\) é a soluç\tilde{a}o exata de um problema levemente perturbado.

Backward error é poderoso porque permite garantias:
“o algoritmo resolveu exatamente um problema próximo do original”.



---
# Módulo 9 — O “erro do erro”

**Mensagem central:** quando você calcula o erro no computador, esse cálculo também é aproximado.

Logo, em geral, você n\tilde{a}o conhece o erro com exatid\tilde{a}o; você conhece uma **estimativa**.


In [ ]:

getcontext().prec = 120

x_real = Decimal("1") / Decimal("7")     # referência (muito precisa)
x_float = float(1/7)

# erro "computado" em float (também sofre arredondamento)
erro_float = abs(x_float - float(x_real))

# erro estimado com Decimal (mais estável como referência aqui)
erro_decimal = abs(x_real - Decimal(str(x_float)))

float(erro_float), erro_decimal


In [ ]:

erro_do_erro = abs(float(erro_decimal) - erro_float)
erro_do_erro



**Discuss\tilde{a}o:**
- Por que o `erro_do_erro` n\tilde{a}o é zero?
- O que isso significa para a ideia de “medir o erro”?
- O que podemos fazer no lugar? (resposta: referências melhores e cotas/limitantes)



---
# Módulo 10 (Opcional) — Acumulaç\tilde{a}o de erros em somas

**Pergunta-guia:** somar muitos termos “pequenos” em float pode perder informaç\tilde{a}o?

Vamos comparar:
1) soma ingênua (ordem natural),
2) soma com Kahan (compensada),
3) soma pareada (pairwise),
4) referência de alta precis\tilde{a}o.


In [ ]:

import random

def soma_ingenua(xs):
    s = 0.0
    for x in xs:
        s += x
    return s

def soma_kahan(xs):
    s = 0.0
    c = 0.0  # compensação
    for x in xs:
        y = x - c
        t = s + y
        c = (t - s) - y
        s = t
    return s

def soma_pareada(xs):
    xs = list(xs)
    if not xs:
        return 0.0
    while len(xs) > 1:
        ys = []
        for i in range(0, len(xs), 2):
            if i+1 < len(xs):
                ys.append(xs[i] + xs[i+1])
            else:
                ys.append(xs[i])
        xs = ys
    return xs[0]

def soma_decimal(xs, prec=120):
    getcontext().prec = prec
    s = Decimal("0")
    for x in xs:
        s += Decimal(str(x))
    return s

# experimento: um termo grande + muitos pequenos
random.seed(0)
N = 200000
xs = [1.0] + [1e-10]*N

s1 = soma_ingenua(xs)
s2 = soma_kahan(xs)
s3 = soma_pareada(xs)
sref = float(soma_decimal(xs, prec=200))

(s1, s2, s3, sref)


In [ ]:

Ea1, Er1 = abs(s1 - sref), abs(s1 - sref)/abs(sref)
Ea2, Er2 = abs(s2 - sref), abs(s2 - sref)/abs(sref)
Ea3, Er3 = abs(s3 - sref), abs(s3 - sref)/abs(sref)

{"ingênua": (Ea1, Er1), "Kahan": (Ea2, Er2), "pareada": (Ea3, Er3)}



**Discuss\tilde{a}o sugerida:**
- Por que a ordem de soma importa?
- Como a soma compensada “recupera” dígitos?
- Isso é um exemplo de melhoria de **estabilidade algorítmica** sem mudar o problema.



---
# 11. Exercícios finais (com espaço para resposta)

> **Instruções**: responda diretamente nas células abaixo.  
> Quando for solicitado “explique”, priorize *argumento* + *evidência numérica* (print/plot) + *conclus\tilde{a}o*.

## Exercício 1 — Erro absoluto vs relativo (escala)
1. Considere as aproximações abaixo para os valores verdadeiros.
2. Calcule erro absoluto e relativo.
3. Discuta qual métrica é mais informativa em cada caso.

Casos:
- (A) \(x=10^6\), \(\tilde{x}=10^6+1\)
- (B) \(x=10^{-6}\), \(\tilde{x}=2\cdot 10^{-6}\)
- (C) \(x=0\), \(\tilde{x}=10^{-12}\)  (comente o erro relativo)

**Resposta (código + explicaç\tilde{a}o):**


In [ ]:
import math

# TODO: implemente aqui e comente seus resultados



## Exercício 2 — Quando a subtraç\tilde{a}o “piora” (investigaç\tilde{a}o)
Considere \(d_k = 1 - (1 - 10^{-k})\) para \(k=1,\dots,20\).

1. Compute \(d_k\) em `float` e em alta precis\tilde{a}o (`decimal` com `prec=80`).
2. Faça uma tabela com: \(k\), `d_float`, `d_decimal`, erro absoluto e erro relativo.
3. Identifique a partir de qual \(k\) o `float` deixa de representar corretamente o resultado (interprete!).

**Resposta (código + observações):**


<details>
<summary><strong>Rubrica (professor)</strong></summary>

- **Tabela completa (35%)**: k, d_float, d_decimal, erro abs/rel.
- **Diagnóstico (45%)**: identifica o “ponto de quebra” (quando 1 - (1-10^{-k}) vira 0 em float) e explica via espaçamento/ulp perto de 1.
- **Discuss\tilde{a}o (20%)**: conecta com cancelamento e epsilon (ordem de 1e-16).

</details>

In [ ]:
from decimal import Decimal, getcontext
import pandas as pd

# TODO: implemente aqui



## Exercício 3 — Epsilon de máquina na prática
1. Obtenha `eps = np.finfo(float).eps`.
2. Verifique numericamente os fatos:
   - `1.0 + eps != 1.0`
   - `1.0 + eps/2 == 1.0`
3. Explique por que isso acontece em termos de arredondamento.

**Resposta:**


<details>
<summary><strong>Rubrica (professor)</strong></summary>

- **Verificações (40%)**: testes com eps e eps/2 corretos.
- **Explicaç\tilde{a}o (60%)**: descreve arredondamento e representabilidade; menciona que eps é o passo mínimo para “mudar” 1.0 em double.

</details>

In [ ]:
import numpy as np

# TODO: implemente aqui e explique



## Exercício 4 — Reformulando para evitar cancelamento
Considere a express\tilde{a}o:

\[ f(x) = \sqrt{1+x} - 1 \]

1. Avalie `f(x)` em `float` para \(x = 10^{-k}\), \(k=1,\dots,16\).
2. Encontre uma forma *algebricamente equivalente* que evite subtraç\tilde{a}o catastrófica.
   - Dica: racionalize o numerador.
3. Compare os resultados (erro relativo), usando alta precis\tilde{a}o (`decimal` ou `mpmath`) como referência.

**Resposta:**


<details>
<summary><strong>Rubrica (professor)</strong></summary>

- **Reescrita correta (35%)**: f(x)=x/(sqrt(1+x)+1).
- **Comparaç\tilde{a}o (45%)**: evidencia numericamente que a forma racionalizada é mais precisa para x pequeno.
- **Referência (20%)**: usa Decimal/mpmath como referência e discute erro relativo.

</details>

In [ ]:
import math
from decimal import Decimal, getcontext
import numpy as np
import pandas as pd

# TODO: implemente aqui



## Exercício 5 — Condicionamento vs estabilidade (conceitual)
Responda em texto (pode incluir exemplos numéricos):

1. O que significa um *problema* ser mal condicionado?
2. O que significa um *algoritmo* ser instável?
3. Dê um exemplo (do notebook ou seu) em que:
   - o problema é bem condicionado, mas um algoritmo pode ser instável; **ou**
   - o problema é mal condicionado, e mesmo um algoritmo estável n\tilde{a}o “salva” a precis\tilde{a}o.

**Resposta (texto):**


<details>
<summary><strong>Rubrica (professor)</strong></summary>

- **Definições (40%)**: condicionamento vs estabilidade bem distinguidos.
- **Exemplo (40%)**: exemplo coerente (p.ex., soma ingênua vs Kahan; ou subtraç\tilde{a}o próxima como mal condicionado).
- **Argumentaç\tilde{a}o (20%)**: texto claro, com justificativa.

</details>

> *(Escreva sua resposta aqui em Markdown. Você pode editar esta célula.)*


## Exercício 6 — Fórmula quadrática (verificaç\tilde{a}o)
Use \(a=1\), \(b=10^8\), \(c=1\).

1. Calcule as duas raízes usando a fórmula “direta”.
2. Calcule as raízes usando uma reformulaç\tilde{a}o estável (como no notebook).
3. Verifique a consistência por substituiç\tilde{a}o: compute \(ax^2+bx+c\) para cada raiz e compare os resíduos.

**Resposta:**


<details>
<summary><strong>Rubrica (professor)</strong></summary>

- **Cálculos (35%)**: raízes direta e estável corretamente implementadas.
- **Resíduos (45%)**: compara ax^2+bx+c e conclui qual é mais confiável.
- **Interpretaç\tilde{a}o (20%)**: menciona cancelamento e estabilidade/condicionamento.

</details>

In [ ]:
import math
import numpy as np
import pandas as pd

# TODO: implemente aqui



---
# 12. Fechamento — O que você deve levar desta aula

- Erros em ponto flutuante s\tilde{a}o inevitáveis: eles vêm da representaç\tilde{a}o finita e do arredondamento.
- **Subtraç\tilde{a}o catastrófica** é um padr\tilde{a}o de instabilidade numérica causado por cancelamento de dígitos significativos.
- **Erro relativo** é a métrica mais sensível para perceber perda de significância (mas exige cuidado quando o valor verdadeiro é ~0).
- **Condicionamento** é propriedade do problema; **estabilidade** é propriedade do algoritmo.
- Boa prática: reescrever fórmulas para evitar cancelamentos e amplificaç\tilde{a}o de erro quando possível.



---
# 13. O erro do erro

Quando calculamos:

$$E = |x - \tilde{x}|$$

esse cálculo também é feito em ponto flutuante.

Logo, o valor computado do erro também sofre arredondamento.

Isso significa que:

- Nunca conhecemos o erro "exato"
- Conhecemos apenas uma estimativa do erro


In [ ]:

import numpy as np
from decimal import Decimal, getcontext

getcontext().prec = 80

x_real = Decimal("1") / Decimal("7")
x_float = float(1/7)

erro_float = abs(x_float - float(x_real))
erro_decimal = abs(x_real - Decimal(str(x_float)))

float(erro_float), erro_decimal



Perguntas:

1. Os dois valores coincidem exatamente?
2. O erro calculado em float é ele próprio exato?
3. O que isso nos diz sobre o que realmente sabemos?


In [ ]:

erro_do_erro = abs(float(erro_decimal) - erro_float)
erro_do_erro



# 14. Modelo Matemático de Aritmética de Ponto Flutuante

Um modelo clássico assume que:

$$\operatorname{fl}(x \circ y) = (x \circ y)(1 + \delta)$$

onde:

- ∘ é uma operaç\tilde{a}o (+, −, ×, ÷)
- $$|\delta| \le \varepsilon_{\mathrm{mach}}$$
- ε_mach é o epsilon da máquina

Isso significa que cada operaç\tilde{a}o introduz um erro relativo pequeno,
mas inevitável.



# 15. Podemos limitar o erro?

Mesmo que n\tilde{a}o saibamos o erro exato,
muitas vezes conseguimos provar que ele está limitado por um valor L.

Exemplo simples:

Considere o cálculo da soma:

S = x + y

Sabemos pelo modelo que:

fl(x + y) = (x + y)(1 + δ)

com $$|\delta| \le \varepsilon_{\mathrm{mach}}$$.

Logo:

|erro relativo| ≤ ε_mach.

Ou seja: sabemos que o erro n\tilde{a}o ultrapassa ε_mach.


In [ ]:

import numpy as np

eps = np.finfo(float).eps

x = 1.234567
y = 9.876543

s_float = x + y
s_real = float(Decimal(str(x)) + Decimal(str(y)))

erro_rel = abs(s_float - s_real)/abs(s_real)

erro_rel, eps



Observe que o erro relativo observado é da ordem de epsilon da máquina.

Isso confirma a estimativa teórica.



# 16. Forward Error vs Backward Error

Forward error:
|\tilde{x} - x|

Backward error:
O resultado computado é a soluç\tilde{a}o exata de um problema levemente perturbado.

Um algoritmo é backward estável se:
ele resolve exatamente um problema próximo do original.

Isso é extremamente importante em Métodos Numéricos,
pois mesmo que o erro exato seja desconhecido,
podemos garantir que ele está dentro de uma perturbaç\tilde{a}o aceitável.



# 17. Exercício — Garantindo um limite superior

Considere a funç\tilde{a}o:

f(x) = 2x

1. Usando o modelo $$\operatorname{fl}(x \circ y) = (x \circ y)(1 + \delta)$$,
   mostre que o erro relativo de calcular 2x em ponto flutuante
   é limitado por ε_mach.

2. Teste numericamente para diferentes valores de x
   e compare o erro relativo com ε_mach.

3. Conclua: mesmo sem saber o erro exato,
   conseguimos garantir um limite superior?


In [ ]:

import numpy as np
from decimal import Decimal, getcontext

getcontext().prec = 80
eps = np.finfo(float).eps

def teste(x):
    real = float(Decimal(str(2))*Decimal(str(x)))
    comp = 2.0*x
    return abs(comp - real)/abs(real)

for val in [1, 1e10, 1e-10, np.pi]:
    print(val, teste(val), " <= eps?", teste(val) <= eps)


---
# Módulo 2 — Medidas de erro (e suas limitações)

Nesta etapa vamos estudar, passo a passo, como medir a diferença entre um valor “verdadeiro” e um valor aproximado.

Suponha que:

- o valor verdadeiro seja $x$;
- a aproximaç\tilde{a}o calculada seja $\tilde{x}$.

Queremos responder à pergunta:

> **Qu\tilde{a}o boa é a aproximaç\tilde{a}o $\tilde{x}$ para $x$?**

Há duas medidas iniciais muito importantes.

## 2.1 Erro absoluto

O **erro absoluto** é definido por

$$
E_a = |x - \tilde{x}|.
$$

Ele mede a distância direta entre o valor verdadeiro e a aproximaç\tilde{a}o.

### Interpretaç\tilde{a}o
- Se $E_a$ é pequeno, ent\tilde{a}o $\tilde{x}$ está perto de $x$ em termos absolutos.
- Mas “pequeno” depende da escala do problema.

Por exemplo:
- errar por $1$ unidade pode ser insignificante quando $x \approx 10^6$;
- mas pode ser enorme quando $x \approx 10^{-6}$.

## 2.2 Erro relativo

O **erro relativo** é definido por

$$
E_r = \frac{|x - \tilde{x}|}{|x|},
$$

desde que $x \neq 0$.

Essa medida compara o erro com o tamanho do próprio valor verdadeiro.

### Interpretaç\tilde{a}o
- O erro relativo diz “qual fraç\tilde{a}o do valor verdadeiro foi perdida”.
- Ele é especialmente útil quando queremos comparar erros em escalas diferentes.

## 2.3 Uma observaç\tilde{a}o importante

Nem sempre o erro relativo é apropriado.

Se $x=0$ ou se $x$ é muito próximo de zero, a express\tilde{a}o

$$
\frac{|x-\tilde{x}|}{|x|}
$$

fica indefinida ou extremamente sensível.

Portanto:
- **erro absoluto** é sempre definido;
- **erro relativo** é muito útil, mas exige cuidado perto de zero.


In [ ]:
x = math.pi
x_aprox = 3.14

Ea = erro_abs(x, x_aprox)
Er = erro_rel(x, x_aprox)

Ea, Er


## 2.4 Primeiro exemplo guiado

Considere $x=\pi$ e $\tilde{x}=3.14$.

O computador nos devolve:
- erro absoluto $E_a = |x-\tilde{x}|$;
- erro relativo $E_r = |x-\tilde{x}|/|x|$.

### Perguntas
1. O valor do erro absoluto, sozinho, já permite dizer se a aproximaç\tilde{a}o é boa?
2. O erro relativo ajuda mais?
3. Em que tipo de problema o erro relativo seria mais informativo do que o absoluto?


## 2.5 Comparando diferentes escalas

Agora vamos olhar três situações.

### Caso A
$$
x = 10^6, \qquad \tilde{x} = 10^6 + 1.
$$

### Caso B
$$
x = 10^{-6}, \qquad \tilde{x} = 2\cdot 10^{-6}.
$$

### Caso C
$$
x = 0, \qquad \tilde{x} = 10^{-12}.
$$

Antes de calcular no Python, tente prever:

- Em qual caso o erro absoluto parece pequeno?
- Em qual caso o erro relativo parece mais revelador?
- Em qual caso o erro relativo deixa de ser uma boa ideia?


In [ ]:
casos = {
    "A": (1e6, 1e6 + 1),
    "B": (1e-6, 2e-6),
    "C": (0.0, 1e-12),
}

for nome, (x, x_aprox) in casos.items():
    Ea = erro_abs(x, x_aprox)
    Er = erro_rel(x, x_aprox)
    print(f"Caso {nome}:")
    print(f"  x       = {x}")
    print(f"  x_aprox = {x_aprox}")
    print(f"  E_abs   = {Ea}")
    print(f"  E_rel   = {Er}")
    print()


## 2.6 Discuss\tilde{a}o dos casos

### Caso A
Temos erro absoluto igual a $1$.
Isso parece grande à primeira vista, mas o valor verdadeiro é da ordem de $10^6$.
Portanto, o erro relativo é muito pequeno.

### Caso B
Temos um erro absoluto muito pequeno, da ordem de $10^{-6}$.
Mesmo assim, o erro relativo é grande, porque o valor verdadeiro também é muito pequeno.

### Caso C
O erro absoluto faz sentido:
$$
E_a = |0 - 10^{-12}| = 10^{-12}.
$$

Mas o erro relativo n\tilde{a}o é adequado, porque exigiria divis\tilde{a}o por zero.

## 2.7 Conclus\tilde{a}o desta etapa

As duas medidas s\tilde{a}o úteis, mas respondem a perguntas diferentes:

- **Erro absoluto**: “de quanto erramos?”
- **Erro relativo**: “quanto esse erro representa em comparaç\tilde{a}o com o tamanho do valor verdadeiro?”

Em Métodos Numéricos, muitas vezes o erro relativo é mais informativo, mas nunca devemos usá-lo mecanicamente.


## 2.8 Mais um exemplo: mesmo erro absoluto, significados muito diferentes

Compare os pares:

$$
1000.0 \quad \text{e} \quad 1000.01
$$

e

$$
0.00100 \quad \text{e} \quad 0.01100.
$$

Nos dois casos, o erro absoluto é $0.01$.
Mesmo assim, o erro relativo muda drasticamente.


In [ ]:
pares = [
    ("Par 1", 1000.0, 1000.01),
    ("Par 2", 0.00100, 0.01100),
]

for nome, x, x_aprox in pares:
    Ea = erro_abs(x, x_aprox)
    Er = erro_rel(x, x_aprox)
    print(f"{nome}: Ea={Ea}, Er={Er}")


## 2.9 Espaço para resposta do aluno

Escreva, com suas palavras:

1. Qual é a diferença conceitual entre erro absoluto e erro relativo?
2. Por que o erro relativo pode ser mais útil em muitos contextos?
3. Por que ele pode ser inadequado quando o valor verdadeiro é zero ou muito pequeno?


> **Resposta do aluno:** edite esta célula Markdown com sua explicaç\tilde{a}o.

---
# Apêndice — Aprofundamentos e módulos opcionais

Os módulos a seguir s\tilde{a}o de aprofundamento.  
Eles podem ser usados na mesma aula, se houver tempo, ou ficar como leitura e experimentaç\tilde{a}o adicional.


---
## Checklist de submiss\tilde{a}o

Antes de entregar seu caderno, verifique se você:

- executou todas as células relevantes;
- preencheu as respostas em Markdown quando solicitado;
- incluiu código, tabelas ou gráficos quando o exercício pediu evidência;
- escreveu conclusões com suas próprias palavras.
